In [41]:
import pandapower as pp

net = pp.networks.case14()
net.bus_geodata.coords = [(x,y) for x,y in zip(net.bus_geodata.x,net.bus_geodata.y)]
pp.runpp(net)


In [50]:
import plotly.graph_objects as go
import networkx as nx

pos = net.bus_geodata.coords

# Asumiendo que estos son los datos que tienes
valores_nodos = net.res_bus.vm_pu  # Sustituye con tu JSON real
valores_lineas = list(net.res_line.loading_percent.values) + list(net.res_trafo.loading_percent.values)   # Sustituye con tu lista real

# Convertir las tuplas a listas para permitir la modificación
edge_trace_lat = []
edge_trace_lon = []
edge_trace_hovertext = []  # Para almacenar la información que se mostrará al pasar el mouse

# Añadir atributos a las líneas
for i, edge in enumerate(net.line.iterrows()):
    edge = edge[1]
    x0, y0 = pos[edge.from_bus]
    x1, y1 = pos[edge.to_bus]
    edge_trace_lat += [y0, y1, None]
    edge_trace_lon += [x0, x1, None]
    edge_trace_hovertext.append(f'Line {i+1} - Valor: {valores_lineas[i]}')

# Añadir transformadores (similar a las líneas)
for i, edge in enumerate(net.trafo.iterrows()):
    edge = edge[1]
    x0, y0 = pos[edge.lv_bus]
    x1, y1 = pos[edge.hv_bus]
    edge_trace_lat += [y0, y1, None]
    edge_trace_lon += [x0, x1, None]
    edge_trace_hovertext.append(f'Transformador {i+1} - Valor: {valores_lineas[i]}')

# Crear las trazas de las conexiones (edges)
edge_trace = go.Scattermapbox(
    lat=edge_trace_lat,
    lon=edge_trace_lon,
    mode='lines',
    line=dict(width=2, color='blue', opacity=valores_lineas),  # Cambia la opacidad según el valor
    hovertext=edge_trace_hovertext,  # Añade la información al pasar el mouse
    hoverinfo='text'
)

# Crear las trazas de los nodos
node_trace = go.Scattermapbox(
    lat=[pos[nodo][1] for nodo in net.bus.index],
    lon=[pos[nodo][0] for nodo in net.bus.index],
    mode='markers+text',
    marker=dict(
        size=10,
        color=[f'rgba({int(255 * valores_nodos[nodo])}, 0, {255 - int(255 * valores_nodos[nodo])}, 1)' for nodo in net.bus.index]  # Escala de color de azul a rojo
    ),
    text=[f'Bus {nodo}' for nodo in net.bus.index],
    textposition="top right",
    hovertext=[f'Bus {nodo} - Valor: {valores_nodos[nodo]}' for nodo in net.bus.index],  # Añade la información al pasar el mouse
    hoverinfo='text'
)

# Configuración del mapa
fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(
                    mapbox_style="open-street-map",
                    mapbox=dict(
                        center=dict(lat=0, lon=0),
                        zoom=12
                    ),
                    showlegend=False,
                    margin=dict(l=0, r=0, t=0, b=0)
                ))

fig.show()

ValueError: Invalid property specified for object of type plotly.graph_objs.scattermapbox.Line: 'opacity'

Did you mean "width"?

    Valid properties:
        color
            Sets the line color.
        width
            Sets the line width (in px).
        
Did you mean "width"?

Bad property path:
opacity
^^^^^^^

In [52]:
import plotly.express as px

pos = net.bus_geodata.coords

# Assuming these are the data you have
valores_nodos = net.res_bus.vm_pu  # Substitute with your actual JSON
valores_lineas = list(net.res_line.loading_percent.values) + list(net.res_trafo.loading_percent.values)  # Substitute with your actual list

# Convert tuples to lists to allow modification
edge_traces = []  # List to store line traces

# Add attributes to lines
for i, edge in enumerate(net.line.iterrows()):
    edge = edge[1]
    x0, y0 = pos[edge.from_bus]
    x1, y1 = pos[edge.to_bus]
    opacity_value = valores_lineas[i] / 100.0  # Normalize the value for opacity (0-1)
    width_value = 2  # You can adjust this or make it vary based on a value if needed

    edge_trace = go.Scattermapbox(
        lat=[y0, y1],
        lon=[x0, x1],
        mode='lines',
        line=dict(width=width_value, color='blue', opacity=opacity_value),
        hovertext=f'Line {i+1} - Value: {valores_lineas[i]}',
        hoverinfo='text'
    )

    edge_traces.append(edge_trace)

# Add transformers (similar to lines)
for i, edge in enumerate(net.trafo.iterrows()):
    edge = edge[1]
    x0, y0 = pos[edge.lv_bus]
    x1, y1 = pos[edge.hv_bus]
    opacity_value = valores_lineas[i + len(net.line)] / 100.0  # Adjust the index for transformers

    edge_trace = go.Scattermapbox(
        lat=[y0, y1],
        lon=[x0, x1],
        mode='lines',
        line=dict(width=width_value, color='blue', opacity=opacity_value),
        hovertext=f'Transformer {i+1} - Value: {valores_lineas[i + len(net.line)]}',
        hoverinfo='text'
    )

    edge_traces.append(edge_trace)

# Create traces for nodes with adjusted colors
node_colors = [
    f'rgba({int(255 * valor)}, 0, {255 - int(255 * valor)}, 1)'
    for valor in valores_nodos
]

node_trace = go.Scattermapbox(
    lat=[pos[nodo][1] for nodo in net.bus.index],
    lon=[pos[nodo][0] for nodo in net.bus.index],
    mode='markers+text',
    marker=dict(
        size=10,
        color=node_colors  # Assign calculated colors
    ),
    text=[f'Bus {nodo}' for nodo in net.bus.index],
    textposition="top right",
    hovertext=[f'Bus {nodo} - Value: {valores_nodos[nodo]}' for nodo in net.bus.index],  # Add hover info
    hoverinfo='text'
)

# Map configuration
fig = go.Figure(data=edge_traces + [node_trace],
                layout=go.Layout(
                    mapbox_style="open-street-map",
                    mapbox=dict(
                        center=dict(lat=0, lon=0),
                        zoom=12
                    ),
                    showlegend=False,
                    margin=dict(l=0, r=0, t=0, b=0)
                ))

fig.show()


ValueError: Invalid property specified for object of type plotly.graph_objs.scattermapbox.Line: 'opacity'

Did you mean "width"?

    Valid properties:
        color
            Sets the line color.
        width
            Sets the line width (in px).
        
Did you mean "width"?

Bad property path:
opacity
^^^^^^^

In [57]:
import plotly.express as px

geo_df = net.bus_geodata


fig = px.scatter_mapbox(geo_df,
                        lat=geo_df.y,
                        lon=geo_df.x,
                        hover_name="coords",
                        zoom=1)

fig.show()

In [64]:
net.bus_geodata

,x,y,coords
0,0.0,0.0,NaN
1,0.0,-1.0,NaN
2,0.0,-2.0,NaN
3,0.0,-3.0,NaN


In [91]:
net.bus_geodata

,x,y,coords
0,0.0,0.0,"(0.0, 0.0)"
1,0.0,-1.0,"(0.0, -1.0)"
2,0.0,-2.0,"(0.0, -2.0)"
3,0.0,-3.0,"(0.0, -3.0)"


In [97]:
import pandapower as pp
import pandapower.networks as nw
import plotly.express as px
import pandas as pd


net = nw.simple_four_bus_system()
net.bus_geodata.coords = [(x,y) for x,y in zip(net.bus_geodata.x,net.bus_geodata.y)]
pp.runpp(net)
# Extracting bus data with vm_pu
bus_data = net.bus[['name']].copy()
bus_data["vm_pu"] = net.res_bus["vm_pu"]
bus_data["geodata"] = net.bus_geodata.coords
bus_data['bus'] = bus_data.index
bus_data['size'] = 10

# Convert geodata to separate latitude and longitude
bus_data['lon'] = bus_data['geodata'].apply(lambda x: x[0])
bus_data['lat'] = bus_data['geodata'].apply(lambda x: x[1])

# Extracting line data
line_data = net.line[['from_bus', 'to_bus']].copy()
line_data = line_data.merge(bus_data[['bus', 'lon', 'lat']], left_on='from_bus', right_on='bus')
line_data = line_data.merge(bus_data[['bus', 'lon', 'lat']], left_on='to_bus', right_on='bus', suffixes=('_from', '_to'))
line_data["load"] = net.res_line.loading_percent

# Plot buses on map with color scaled by vm_pu
fig = px.scatter_mapbox(bus_data, lat="lat", lon="lon", hover_name="name", hover_data=[],
                        color="vm_pu",size="size",size_max=10, color_continuous_scale=[[0, "blue"], [0.5, "green"], [1, "red"]],
                        range_color=[0.9, 1.1], zoom=6, height=600,
                        title="Pandapower Network - Bus Locations on Map")

# Add lines to represent connections between buses
for _, row in line_data.iterrows():
    fig.add_trace(px.line_mapbox(lat=[row['lat_from'], row['lat_to']],
                                 lon=[row['lon_from'], row['lon_to']],
                                 ).data[0])

# Update layout for mapbox style and settings
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(showlegend=False)

fig.show()

ValueError: String or int arguments are only possible when a DataFrame or an array is provided in the `data_frame` argument. No DataFrame was provided, but argument 'color' is of type str or int.

In [94]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.027611,0.013328,-0.025713,-0.013088,0.001899,0.000241,0.044405,0.044408,0.044408,0.996608,-150.208127,0.93776,-149.007459,31.273098
1,0.015713,0.008088,-0.015000,-0.008000,0.000713,0.000088,0.027200,0.027203,0.027203,0.937760,-149.007459,0.90200,-148.184087,19.157249
